In [1]:
from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse

PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data" / "processed" / "scgpt_input"
OUT_DIR = PROJECT_DIR / "data" / "processed" / "scfoundation_input"
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [10]:
from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse

def adata_to_dataframe(adata):
    X = adata.X
    if sparse.issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float32)
    df = pd.DataFrame(
        X,
        index=adata.obs_names.astype(str),
        columns=adata.var_names.astype(str)
    )
    return df
def main_gene_selection_local(X_df, gene_list):
    """
    Align dataframe columns to the target gene list.
    Cells in rows, genes in columns.
    Missing genes are filled with 0.
    Extra genes are dropped.
    """
    X_df = X_df.copy()
    X_df.columns = X_df.columns.astype(str)

    overlap_genes = [g for g in gene_list if g in X_df.columns]
    missing_genes = [g for g in gene_list if g not in X_df.columns]

    # keep overlap
    X_df = X_df.loc[:, overlap_genes]

    # fill missing genes with 0
    if len(missing_genes) > 0:
        fill_df = pd.DataFrame(
            0,
            index=X_df.index,
            columns=missing_genes,
            dtype=np.float32
        )
        X_df = pd.concat([X_df, fill_df], axis=1)

    # reorder to target gene list
    X_df = X_df.reindex(columns=gene_list, fill_value=0)

    var = pd.DataFrame(index=gene_list)

    return X_df, missing_genes, var



In [12]:
SCFOUNDATION_MODEL_DIR = Path(PROJECT_DIR / "models" / "scFoundation") 

gene_list_df = pd.read_csv(
    SCFOUNDATION_MODEL_DIR / "OS_scRNA_gene_index.19264.tsv",
    sep="\t"
)
gene_list = list(gene_list_df["gene_name"].astype(str))

print("Target genes:", len(gene_list))
print(gene_list[:5])


Target genes: 19264
['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2']


In [18]:
import numpy as np

def stratified_subsample_adata(adata, label_col="label", max_per_class=None, random_state=42):
    if max_per_class is None:
        raise ValueError("max_per_class must be provided as a dict")

    rng = np.random.default_rng(random_state)
    keep_idx = []

    labels = adata.obs[label_col].astype(str)

    for lab, max_n in max_per_class.items():
        idx = np.where(labels.values == lab)[0]
        if len(idx) <= max_n:
            keep_idx.extend(idx.tolist())
        else:
            keep_idx.extend(rng.choice(idx, size=max_n, replace=False).tolist())

    keep_idx = np.array(sorted(keep_idx))
    return adata[keep_idx].copy()


In [19]:
from pathlib import Path
import scanpy as sc

PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data" / "processed" / "scgpt_input"

adata_ref_full = sc.read_h5ad(DATA_DIR / "pancreas_ref_scgpt_input.h5ad")

adata_ref_sub = stratified_subsample_adata(
    adata_ref_full,
    label_col="label",
    max_per_class={
        "alpha": 3000,
        "beta": 3000,
        "delta": 2889,
        "pp": 121
    },
    random_state=42
)

print(adata_ref_sub)
print(adata_ref_sub.obs["label"].value_counts())


AnnData object with n_obs × n_vars = 9010 × 24233
    obs: 'sample', 'louvain_anno_broad', 'louvain_anno_fine', 'donor_id', 'BMI', 'HbA1c', 'insulin_content', 'glucose_SI', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'n_genes', 'n_counts', 'mt_frac', 'size_factors', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'cell_type_original', 'label', 'dataset_name', 'dataset_role', 'cohort', 'n_genes_by_counts', 'total_counts'
    var: 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'gene_symbol', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'ensembl_id'
    uns: 'batch_condition', 'citation', 'default_embedding', 'organism', '

In [20]:
from pathlib import Path
import pandas as pd

RESULT_DIR = Path("results") / "scgpt"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "cell_id": adata_ref_sub.obs_names.astype(str),
    "label": adata_ref_sub.obs["label"].astype(str).values
}).to_csv(RESULT_DIR / "ref_subset_cells.csv", index=False)


In [21]:
def export_for_scfoundation(adata, out_csv, gene_list):
    X_df = adata_to_dataframe(adata)
    X_df_aligned, missing_genes, var = main_gene_selection_local(X_df, gene_list)

    print(f"\n{out_csv.name}")
    print("aligned shape:", X_df_aligned.shape)
    print("missing genes filled:", len(missing_genes))

    X_df_aligned.to_csv(out_csv, index=False)
    return X_df_aligned


In [23]:
from pathlib import Path
import scanpy as sc

DATA_DIR = Path("data/processed/scgpt_input")
OUT_DIR = Path("data/processed/scfoundation_input")
OUT_DIR.mkdir(parents=True, exist_ok=True)

adata_ref_sub = adata_ref_sub.copy()  
adata_control = sc.read_h5ad(DATA_DIR / "pancreas_control_scgpt_input.h5ad")
adata_aab = sc.read_h5ad(DATA_DIR / "pancreas_aab_scgpt_input.h5ad")
adata_t1d = sc.read_h5ad(DATA_DIR / "pancreas_t1d_scgpt_input.h5ad")

export_for_scfoundation(adata_ref_sub, OUT_DIR / "pancreas_ref_subset.csv", gene_list)
export_for_scfoundation(adata_control, OUT_DIR / "pancreas_control.csv", gene_list)
export_for_scfoundation(adata_aab, OUT_DIR / "pancreas_aab.csv", gene_list)
export_for_scfoundation(adata_t1d, OUT_DIR / "pancreas_t1d.csv", gene_list)



pancreas_ref_subset.csv
aligned shape: (9010, 19264)
missing genes filled: 2170

pancreas_control.csv
aligned shape: (8669, 19264)
missing genes filled: 2170

pancreas_aab.csv
aligned shape: (13122, 19264)
missing genes filled: 2170

pancreas_t1d.csv
aligned shape: (3267, 19264)
missing genes filled: 2170


,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
AAACCTGAGTAGCGGT-20,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCTGTCATGTCTT-20,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAAGATGGTCAAACTC-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
AAAGCAAAGTGTACCT-20,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAAGCAACAAGTAGTA-20,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCTCATATCTTGTATC-25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TGCCCATAGCACCGCT-25,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
TGGCCAGTCCGAACGC-25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
TGGTTAGGTGTGGTTT-25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path("data/processed/scfoundation_input")

control_df = pd.read_csv(OUT_DIR / "pancreas_control.csv")
control_200 = control_df.iloc[:200].copy()
control_200.to_csv(OUT_DIR / "pancreas_control_200.csv", index=False)

print(control_200.shape)


(200, 19264)


In [25]:
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path
from scipy import sparse

PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data" / "processed" / "scgpt_input"
OUT_DIR = PROJECT_DIR / "data" / "processed" / "scfoundation_input"
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [26]:
def stratified_subsample_adata(adata, label_col="label", max_per_class=None, random_state=42):
    rng = np.random.default_rng(random_state)
    labels = adata.obs[label_col].astype(str).values
    keep_idx = []

    for lab, max_n in max_per_class.items():
        idx = np.where(labels == lab)[0]
        if len(idx) <= max_n:
            keep_idx.extend(idx.tolist())
        else:
            keep_idx.extend(rng.choice(idx, size=max_n, replace=False).tolist())

    keep_idx = np.array(sorted(keep_idx))
    return adata[keep_idx].copy()

adata_ref = sc.read_h5ad(DATA_DIR / "pancreas_ref_scgpt_input.h5ad")
adata_control = sc.read_h5ad(DATA_DIR / "pancreas_control_scgpt_input.h5ad")
adata_aab = sc.read_h5ad(DATA_DIR / "pancreas_aab_scgpt_input.h5ad")
adata_t1d = sc.read_h5ad(DATA_DIR / "pancreas_t1d_scgpt_input.h5ad")


In [27]:
adata_ref_pilot = stratified_subsample_adata(
    adata_ref,
    max_per_class={"alpha": 100, "beta": 100, "delta": 100, "pp": 121},
    random_state=42
)

adata_control_pilot = stratified_subsample_adata(
    adata_control,
    max_per_class={"alpha": 50, "beta": 50, "delta": 50, "pp": 50},
    random_state=42
)

adata_aab_pilot = stratified_subsample_adata(
    adata_aab,
    max_per_class={"alpha": 50, "beta": 50, "delta": 50, "pp": 50},
    random_state=42
)

adata_t1d_pilot = stratified_subsample_adata(
    adata_t1d,
    max_per_class={"alpha": 50, "beta": 50, "delta": 50, "pp": 50},
    random_state=42
)

print(adata_ref_pilot.obs["label"].value_counts())
print(adata_control_pilot.obs["label"].value_counts())
print(adata_aab_pilot.obs["label"].value_counts())
print(adata_t1d_pilot.obs["label"].value_counts())


label
pp       121
alpha    100
beta     100
delta    100
Name: count, dtype: int64
label
alpha    50
beta     50
delta    50
pp       50
Name: count, dtype: int64
label
alpha    50
beta     50
delta    50
pp       50
Name: count, dtype: int64
label
alpha    50
beta     50
delta    50
pp       50
Name: count, dtype: int64


In [28]:
RESULT_DIR = PROJECT_DIR / "results" / "scfoundation"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def save_meta(adata, out_csv):
    pd.DataFrame({
        "cell_id": adata.obs_names.astype(str),
        "label": adata.obs["label"].astype(str).values
    }).to_csv(out_csv, index=False)

save_meta(adata_ref_pilot, RESULT_DIR / "ref_pilot_meta.csv")
save_meta(adata_control_pilot, RESULT_DIR / "control_pilot_meta.csv")
save_meta(adata_aab_pilot, RESULT_DIR / "aab_pilot_meta.csv")
save_meta(adata_t1d_pilot, RESULT_DIR / "t1d_pilot_meta.csv")


In [29]:
def export_for_scfoundation(adata, out_csv, gene_list):
    X = adata.X
    if sparse.issparse(X):
        X = X.toarray()
    X_df = pd.DataFrame(
        np.asarray(X, dtype=np.float32),
        index=adata.obs_names.astype(str),
        columns=adata.var_names.astype(str)
    )

    X_df_aligned, missing_genes, var = main_gene_selection_local(X_df, gene_list)

    print(f"\n{out_csv.name}")
    print("aligned shape:", X_df_aligned.shape)
    print("missing genes filled:", len(missing_genes))

    X_df_aligned.to_csv(out_csv, index=False)
    return X_df_aligned


In [30]:
export_for_scfoundation(adata_ref_pilot, OUT_DIR / "pancreas_ref_pilot.csv", gene_list)
export_for_scfoundation(adata_control_pilot, OUT_DIR / "pancreas_control_pilot.csv", gene_list)
export_for_scfoundation(adata_aab_pilot, OUT_DIR / "pancreas_aab_pilot.csv", gene_list)
export_for_scfoundation(adata_t1d_pilot, OUT_DIR / "pancreas_t1d_pilot.csv", gene_list)



pancreas_ref_pilot.csv
aligned shape: (421, 19264)
missing genes filled: 2170

pancreas_control_pilot.csv
aligned shape: (200, 19264)
missing genes filled: 2170

pancreas_aab_pilot.csv
aligned shape: (200, 19264)
missing genes filled: 2170

pancreas_t1d_pilot.csv
aligned shape: (200, 19264)
missing genes filled: 2170


,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
AATCGGTAGGGTGTTG-20,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
ACGGGCTCAGGTGCCT-20,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
ACTGAGTAGGTGTGGT-20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AGAGCGAAGTCGAGTG-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AGTCTTTGTGTTCGAT-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTGACTTCACAACTGT-24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTGGCAATCGTCGTTC-24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAGACCTGTCCCGACA-25,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
ACACCAATCGGAGGTA-25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


python get_embedding.py \
  --task_name pancreas_ref_pilot \
  --input_type singlecell \
  --output_type cell \
  --pool_type all \
  --tgthighres a5 \
  --data_path /Users/zhuqin/Desktop/CMML/ICA2/data/processed/scfoundation_input/pancreas_ref_pilot.csv \
  --save_path /Users/zhuqin/Desktop/CMML/ICA2/results/scfoundation/ \
  --pre_normalized F \
  --version ce \
  --ckpt_name 01B-resolution

python get_embedding.py \
  --task_name pancreas_control_pilot \
  --input_type singlecell \
  --output_type cell \
  --pool_type all \
  --tgthighres a5 \
  --data_path /Users/zhuqin/Desktop/CMML/ICA2/data/processed/scfoundation_input/pancreas_control_pilot.csv \
  --save_path /Users/zhuqin/Desktop/CMML/ICA2/results/scfoundation/ \
  --pre_normalized F \
  --version ce \
  --ckpt_name 01B-resolution

python get_embedding.py \
  --task_name pancreas_aab_pilot \
  --input_type singlecell \
  --output_type cell \
  --pool_type all \
  --tgthighres a5 \
  --data_path /Users/zhuqin/Desktop/CMML/ICA2/data/processed/scfoundation_input/pancreas_aab_pilot.csv \
  --save_path /Users/zhuqin/Desktop/CMML/ICA2/results/scfoundation/ \
  --pre_normalized F \
  --version ce \
  --ckpt_name 01B-resolution

python get_embedding.py \
  --task_name pancreas_t1d_pilot \
  --input_type singlecell \
  --output_type cell \
  --pool_type all \
  --tgthighres a5 \
  --data_path /Users/zhuqin/Desktop/CMML/ICA2/data/processed/scfoundation_input/pancreas_t1d_pilot.csv \
  --save_path /Users/zhuqin/Desktop/CMML/ICA2/results/scfoundation/ \
  --pre_normalized F \
  --version ce \
  --ckpt_name 01B-resolution


In [31]:
import numpy as np
import pandas as pd
from pathlib import Path

RESULT_DIR = Path("results/scfoundation")

ref_embed = np.load(RESULT_DIR / "pancreas_ref_pilot_01B-resolution_singlecell_cell_embedding_a5_resolution.npy")
control_embed = np.load(RESULT_DIR / "pancreas_control_pilot_01B-resolution_singlecell_cell_embedding_a5_resolution.npy")
aab_embed = np.load(RESULT_DIR / "pancreas_aab_pilot_01B-resolution_singlecell_cell_embedding_a5_resolution.npy")
t1d_embed = np.load(RESULT_DIR / "pancreas_t1d_pilot_01B-resolution_singlecell_cell_embedding_a5_resolution.npy")

ref_meta = pd.read_csv(RESULT_DIR / "ref_pilot_meta.csv")
control_meta = pd.read_csv(RESULT_DIR / "control_pilot_meta.csv")
aab_meta = pd.read_csv(RESULT_DIR / "aab_pilot_meta.csv")
t1d_meta = pd.read_csv(RESULT_DIR / "t1d_pilot_meta.csv")

print(ref_embed.shape, control_embed.shape, aab_embed.shape, t1d_embed.shape)
print(ref_meta.shape, control_meta.shape, aab_meta.shape, t1d_meta.shape)


(421, 3072) (200, 3072) (200, 3072) (200, 3072)
(421, 2) (200, 2) (200, 2) (200, 2)


In [32]:
from sklearn.linear_model import LogisticRegression

y_train = ref_meta["label"].astype(str).values
y_control = control_meta["label"].astype(str).values
y_aab = aab_meta["label"].astype(str).values
y_t1d = t1d_meta["label"].astype(str).values

clf = LogisticRegression(
    max_iter=3000,
    multi_class="multinomial",
    class_weight="balanced",
    random_state=42
)

clf.fit(ref_embed, y_train)

pred_control = clf.predict(control_embed)
pred_aab = clf.predict(aab_embed)
pred_t1d = clf.predict(t1d_embed)


/Users/zhuqin/miniconda3/envs/cmml3_scfoundation/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

LABEL_ORDER = ["alpha", "beta", "delta", "pp"]

def evaluate_dataset(name, y_true, y_pred, out_dir):
    acc = accuracy_score(y_true, y_pred)

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_ORDER, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_ORDER, average="weighted", zero_division=0
    )

    metrics_df = pd.DataFrame([{
        "dataset": name,
        "n_cells": len(y_true),
        "accuracy": acc,
        "macro_precision": p_macro,
        "macro_recall": r_macro,
        "macro_f1": f1_macro,
        "weighted_precision": p_weighted,
        "weighted_recall": r_weighted,
        "weighted_f1": f1_weighted
    }])
    metrics_df.to_csv(out_dir / f"{name}_metrics.csv", index=False)

    report = classification_report(
        y_true, y_pred, labels=LABEL_ORDER,
        output_dict=True, zero_division=0
    )
    pd.DataFrame(report).T.to_csv(out_dir / f"{name}_classification_report.csv")

    cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER).to_csv(
        out_dir / f"{name}_confusion_matrix.csv"
    )

    print(f"\n===== {name} =====")
    print(metrics_df.to_string(index=False))

    return metrics_df


In [34]:
all_metrics = []
all_metrics.append(evaluate_dataset("control", y_control, pred_control, RESULT_DIR))
all_metrics.append(evaluate_dataset("aab", y_aab, pred_aab, RESULT_DIR))
all_metrics.append(evaluate_dataset("t1d", y_t1d, pred_t1d, RESULT_DIR))

all_metrics_df = pd.concat(all_metrics, ignore_index=True)
all_metrics_df.to_csv(RESULT_DIR / "all_metrics_summary.csv", index=False)

print("\nSaved overall summary:")
print(all_metrics_df.to_string(index=False))



===== control =====
dataset  n_cells  accuracy  macro_precision  macro_recall  macro_f1  weighted_precision  weighted_recall  weighted_f1
control      200     0.765          0.78867         0.765  0.760454             0.78867            0.765     0.760454

===== aab =====
dataset  n_cells  accuracy  macro_precision  macro_recall  macro_f1  weighted_precision  weighted_recall  weighted_f1
    aab      200     0.765         0.795668         0.765  0.760094            0.795668            0.765     0.760094

===== t1d =====
dataset  n_cells  accuracy  macro_precision  macro_recall  macro_f1  weighted_precision  weighted_recall  weighted_f1
    t1d      200      0.76         0.825415          0.76  0.751298            0.825415             0.76     0.751298

Saved overall summary:
dataset  n_cells  accuracy  macro_precision  macro_recall  macro_f1  weighted_precision  weighted_recall  weighted_f1
control      200     0.765         0.788670         0.765  0.760454            0.788670        